# C1.1 · The agentic offensive workflow, and containing it

**Function C — Red Teaming and Security Research with AI → Red Teaming with AI**  ·  *Both directions*

Builds on **[C1.0 · Start here — what red teaming and research with AI means](https://spbreed.github.io/cyber-commons/lessons/C1.0.html)**.

| | |
|---|---|
| Tools used | CAI, Metasploit, Firecracker, Kimi K2, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Drive a planner/executor pair against a local target and watch the scope guard refuse an out-of-scope host before the request leaves.

**Why a security engineer needs it.** Payload suggestions instead of attack chains — and an offensive loop with no hard scope enforcement, which is an incident with a project plan. The control it builds is: full target context before it swings, and scope enforced at the network layer rather than by a politeness clause in the prompt.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An offensive harness reads only hostile input, by definition: every byte comes from a system you are attacking. It is the most dangerous agent in the building, and the thing that makes running it professional is that scope stops living in the tester's attention.

> **At CyberTravels.** An offensive loop pointed at CyberTravels' staging estate is the most dangerous thing in the building — and the engagement scope has to be enforced below the model, because everything the harness reads comes from the system it is attacking.

## 2 · The framework

```
   recon --> hypothesis --> test --> escalate --> report
        (the loop has not changed; who runs each turn has)

   +--------------------------------------------------+
   |  harness scope check   host in engagement set?   |
   +--------------------------------------------------+
   |  sandbox egress        private/link-local? rate? |
   +--------------------------------------------------+
              two layers, neither of them the model

   everything the harness reads is hostile by design
```

Penetration testing has always been a loop: **recon → hypothesis → test →
escalate → report.** What has changed is who runs each turn.

**Manual (still the baseline).** A human runs `nmap`, reads the output, forms a
hypothesis, tries it. Slow, and the quality is entirely the tester's.

**Scripted.** The recon is automated — Nuclei templates, a Burp scan. The
hypothesis and the escalation are still human. This is where most teams are.

**Semi-autonomous.** An open-weight model reads the recon output and *proposes*
which findings are worth chasing and what to try next. The human approves each
action. The gain is triage speed on a large surface: 400 findings ranked in
minutes rather than a day.

**Autonomous.** The model proposes and the harness executes, within a
pre-approved scope and tool set, verifying its own results. This is real and it
works, and it is also where the engagement becomes a safety problem — because an
agent that has not understood the scope will happily test something outside it
at machine speed.

The professional obligations do not change with autonomy. They get harder,
because scope enforcement can no longer live in the tester's attention — it has
to live in the harness, and then underneath the harness in the network.

That second half is why containment belongs in this lesson rather than in a
later one. An offensive harness has a property no other agent has: **everything
it reads is hostile by design.** Banner strings, error bodies, file contents —
all of it comes from a system you are attacking, which may itself already be
attacker-controlled. Containment there protects three parties at once: the
client (scope and rate limits, so you do not break their production), everyone
else (egress control, so a compromised harness cannot pivot outward), and you
(findings and client data must not leave by a route the agent chooses).

## 3 · The model backend, and the triage it proposes

In [ ]:
# --- model backend: replay by default, real model when you configure one ----
# Nothing here is Anthropic- or vendor-specific beyond one URL and one header
# shape. Standard library only, so the notebook stays self-contained.
import json, os, urllib.error, urllib.request

# The cheapest current model on each side, which is what a lesson needs.
FRONTIER_DEFAULT   = "claude-haiku-4-5-20251001"
OPEN_WEIGHT_DEFAULT = "glm-4.6"
TIMEOUT = 60

def _kaggle_secret(name):
    """On Kaggle, a key lives in Add-ons -> Secrets rather than the environment.

    kaggle_secrets is pre-installed in the Kaggle image and absent everywhere
    else, so the import is guarded and the notebook needs no dependency. It also
    requires the notebook to have internet enabled, which on Kaggle requires a
    phone-verified account - see the note printed below.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def backend():
    """(kind, model). Configuration comes from the environment, never a literal."""
    if os.environ.get("ANTHROPIC_API_KEY") or _kaggle_secret("ANTHROPIC_API_KEY"):
        os.environ.setdefault("ANTHROPIC_API_KEY",
                              os.environ.get("ANTHROPIC_API_KEY")
                              or _kaggle_secret("ANTHROPIC_API_KEY") or "")
        return "frontier", os.environ.get("MODEL", FRONTIER_DEFAULT)
    if os.environ.get("OPENAI_BASE_URL"):
        return "open-weight", os.environ.get("MODEL", OPEN_WEIGHT_DEFAULT)
    return "replay", "deterministic stand-in (no backend configured)"

def _post(url, payload, headers):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"content-type": "application/json", **headers})
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        return json.loads(r.read().decode())

def _anthropic(prompt, system, model, max_tokens, temperature):
    body = {"model": model, "max_tokens": max_tokens, "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]}
    if system:
        body["system"] = system
    headers = {"x-api-key": os.environ["ANTHROPIC_API_KEY"],
               "anthropic-version": "2023-06-01"}
    # An identity-linked key is scoped to a workspace and the API refuses the
    # call without being told which one. A plain organisation key needs nothing
    # here, so the header is only sent when it is set.
    ws = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    if ws:
        headers["anthropic-workspace-id"] = ws
    base = os.environ.get("ANTHROPIC_BASE_URL", "https://api.anthropic.com").rstrip("/")
    out = _post(f"{base}/v1/messages", body, headers)
    return "".join(b.get("text", "") for b in out.get("content", [])).strip()

def _openai_compatible(prompt, system, model, max_tokens, temperature):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    base = os.environ["OPENAI_BASE_URL"].rstrip("/")
    key = os.environ.get("OPENAI_API_KEY", "not-needed")
    out = _post(f"{base}/chat/completions",
                {"model": model, "messages": msgs, "max_tokens": max_tokens,
                 "temperature": temperature},
                {"authorization": f"Bearer {key}"})
    return out["choices"][0]["message"]["content"].strip()

def ask(prompt, *, replay, system=None, max_tokens=512, temperature=0.0):
    """Answer `prompt` with the configured backend, or return `replay`.

    `replay` is required, not optional: a lesson must be able to run offline,
    and the answer it falls back to has to be visible in the source rather than
    invented at runtime.
    """
    kind, model = backend()
    if kind == "replay":
        return replay, kind, model
    try:
        fn = _anthropic if kind == "frontier" else _openai_compatible
        return fn(prompt, system, model, max_tokens, temperature), kind, model
    except (urllib.error.URLError, urllib.error.HTTPError, KeyError, TimeoutError) as e:
        # Print what the API actually said. "failed: 400" costs whoever hits
        # this an hour; the body usually names the exact missing header or
        # parameter, and it never contains the key.
        detail = getattr(e, "code", None) or type(e).__name__
        why = ""
        if hasattr(e, "read"):
            try:
                why = json.loads(e.read().decode()).get("error", {}).get("message", "")
            except Exception:
                why = ""
        print(f"   !! {kind} backend ({model}) failed: {detail}"
              f"{' - ' + why if why else ''}")
        print("      Using the replay, which is labelled as one. No model answered.")
        return replay, "replay", f"{model} unreachable"

_kind, _model = backend()
print(f"model backend : {_kind}")
print(f"model         : {_model}")
if _kind == "replay":
    print()
    print("This lesson runs offline against a deterministic replay, which is why")
    print("it works on a Kaggle kernel with the internet switched off. To run the")
    print("identical code against a real model, set one of:")
    print()
    print("   frontier     export ANTHROPIC_API_KEY=...   # cheapest: " + FRONTIER_DEFAULT)
    print("                (an identity-linked key also needs")
    print("                 ANTHROPIC_WORKSPACE_ID=...)")
    print("   open weight  export OPENAI_BASE_URL=http://localhost:11434/v1 \\")
    print("                       OPENAI_API_KEY=ollama MODEL=glm-4.6")
    print()
    print("   On Kaggle: Add-ons -> Secrets, add ANTHROPIC_API_KEY, and switch")
    print("   Internet on in the notebook settings. Internet requires a")
    print("   phone-verified Kaggle account; without it DNS fails in the kernel")
    print("   and this lesson correctly stays on the replay.")

## 4 · The same lesson, against a real model

Everything below this point runs identically on three backends. Offline it uses
a deterministic replay that is labelled as a replay wherever it appears — never
presented as a model's output. With `ANTHROPIC_API_KEY` set it calls a frontier
model; with `OPENAI_BASE_URL` set it calls any OpenAI-compatible endpoint,
which covers Ollama, vLLM and the hosted open-weight providers.

The point of running it both ways is not that the answers match. It is that
**the lesson's assertion holds either way** — if it only holds against the
replay, the lesson was testing the replay.

In [ ]:
TASK = 'Rank these findings by which to chase first on an authorised engagement, and say why in one clause each.\nF-01 TLS 1.0 enabled on api.target.example\nF-02 /v1/users returns data without auth on api.target.example\nF-06 expired certificate on cdn.partner.example (not in scope)'

REPLAY = '1. F-02 - unauthenticated data endpoint, directly exploitable.\n2. F-01 - needs a downgrade position; no evidence of one here.\n3. F-06 - out of scope, do not touch.'

answer, used, model = ask(TASK, replay=REPLAY,
            system='You triage penetration-test findings. Ranked list, one clause each.',
            max_tokens=300)

print(f"backend used : {used}")
print(f"model        : {model}")
print(f"prompt       : {TASK[:66]}...")
print()
print("answer:")
for line in (answer.splitlines() or [answer]):
    print(f"   {line}")

# Two assertions that must hold on every backend, and one property that is
# reported rather than asserted - a real model failing it is a finding about
# the model, not a broken notebook.
assert answer.strip(), "the configured backend returned nothing"
if used == "replay":
    assert answer == REPLAY, "the offline path must return the replay verbatim"

label, held = ("put the unauthenticated endpoint first", answer.find("F-02") in range(0, 40))
print()
print(f"property checked : {label}")
print(f"held on {used:12s} : {held}")
print()
print("Same code, same assertions, three possible backends. Offline the answer")
print("is the replay and is labelled as one; with a key it is the model's.")

## 5 · Demo — the four generations on the same recon output

Realistic scan output from an authorised engagement against hosts you own. The question at every generation is the same: what do I chase first?

## 6 · Where it breaks — generation 4, and the scope problem

The model's top-ranked item is correct. Its reasoning on F-06 is also correct — *out of scope, do not touch*. Now make it autonomous and remove the human from the loop. What stops it acting on a finding it has correctly identified as out of scope?

Nothing in the model. Its judgement about scope is a *proposal*, on the decision plane, exactly like everything else it produces.

## 7 · The control — and the layer underneath it

The scope check above lives in the harness, which is one process away from the loop it constrains. On an engagement a single control is a single point of failure, and the failure is a professional incident. The same rule therefore gets restated where the agent cannot reach it: the sandbox's own request path.

## What you just proved

Severity sorting puts 2 of 3 exploitable findings in the top 3; model triage puts 3 of 3, and correctly reasons that the partner CDN is out of scope. With the model adversarially convinced that the out-of-scope host is critical, the unenforced harness acts on it and the enforced harness refuses. Underneath the harness the sandbox refuses three requests for three different reasons — rate limit, engagement boundary, and cloud metadata — without consulting the model at all.

## Your turn

Write your engagement scope as a data structure your harness reads, not as a paragraph in a PDF. Then ask what your current tooling would do if a target redirected to a host you were not authorised to touch.

---

**Next → [C1.2 · Red-teaming an agent: designing the campaign](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*